# 0 Data Preprocessing

In this notebook, we transform the raw Electricity Load Diagrams dataset into a format suitable for Machine Learning. 
We will clean the data, engineer time-based features, reshape the dataset from "wide" to "long" format, and create lag/rolling window features to capture historical consumption patterns. Finally, we will export the processed dataset.

## 1. Imports
We import the necessary libraries for data manipulation and date calculations. `dateutil.easter` is used to dynamically calculate the dates of movable Portuguese holidays.

In [ ]:
import pandas as pd
import numpy as np
import datetime
from dateutil import easter
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import sys
import os
sys.path.append(os.path.abspath(".."))

from src.tools import add_temporal_features, get_national_weather, clean_clients, load_raw_data

df = 

In [14]:
# Caricamento raw
df = pd.read_csv("../Datasets/Electricity Dataset.csv", sep=';', decimal=',')
df = df.rename(columns={'Unnamed: 0': 'Date'})
df['Date'] = pd.to_datetime(df['Date'])

# Applichiamo le feature temporali
df = add_temporal_features(df)

# Sanity Check: Vediamo se riconosce il Natale o i Weekend
display(df)

,Date,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,...,MT_366,MT_367,MT_368,MT_369,MT_370,Weekday,Hour,Month,Is_Weekend,Is_Holiday
0,2011-01-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,1,2,True,True
1,2011-01-01 00:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,1,2,True,True
2,2011-01-01 00:45:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,1,2,True,True
3,2011-01-01 01:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,2,2,True,True
4,2011-01-01 01:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,2,2,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140251,2014-12-31 23:00:00,2.538071,22.048364,1.737619,150.406504,85.365854,303.571429,11.305822,282.828283,68.181818,...,5.851375,697.102722,176.961603,651.026393,7621.621622,3,24,13,False,False
140252,2014-12-31 23:15:00,2.538071,21.337127,1.737619,166.666667,81.707317,324.404762,11.305822,252.525253,64.685315,...,9.947338,671.641791,168.614357,669.354839,6702.702703,3,24,13,False,False
140253,2014-12-31 23:30:00,2.538071,20.625889,1.737619,162.601626,82.926829,318.452381,10.175240,242.424242,61.188811,...,9.362200,670.763828,153.589316,670.087977,6864.864865,3,24,13,False,False
140254,2014-12-31 23:45:00,1.269036,21.337127,1.737619,166.666667,85.365854,285.714286,10.175240,225.589226,64.685315,...,4.095963,664.618086,146.911519,646.627566,6540.540541,3,24,13,False,False


## 2. Helper Functions
Here we define the functions to calculate Portuguese national holidays and generate temporal features (like the hour of the day, day of the week, and whether a date is a weekend or holiday). 

Crucially, this function accounts for dynamically movable holidays (like Carnaval and Easter) and manages the 2013-2015 Portuguese government austerity measures (Troika period) which temporarily suspended 4 major public holidays.

In [2]:
def get_holidays(years):
    """
    Generates a set of Portuguese national holiday dates for a list of given years.
    """
    holidays = set()
    for year in years:
        e_day = easter.easter(year)
        good_friday = e_day - datetime.timedelta(days=2)
        corpus_christi = e_day + datetime.timedelta(days=60)
        carnaval = e_day - datetime.timedelta(days=47)

        
        holidays.update([
            datetime.date(year, 1, 1),    # Ano Novo (New Year's Day)
            datetime.date(year, 4, 25),   # Dia da Liberdade (Freedom Day)
            datetime.date(year, 5, 1),    # Dia do Trabalhador (Labor Day)
            datetime.date(year, 6, 10),   # Dia de Portugal (Portugal Day)
            datetime.date(year, 8, 15),   # Assunção de Nossa Senhora (Assumption of Mary)
            datetime.date(year, 12, 8),   # Imaculada Conceição (Immaculate Conception)
            datetime.date(year, 12, 25),  # Natal (Christmas Day)
            good_friday,                  # Sexta-feira Santa (Good Friday)
            e_day,                        # Páscoa (Easter Sunday)
            carnaval                      # Carnaval (Shrove Tuesday: Unofficial but heavily observed)
        ])

        # Holidays suspended by the government (Troika) between 2013 and 2015
        # More infos at https://www.theguardian.com/world/2016/jan/08/portugals-socialist-government-restores-holidays-cut-during-austerity-drive
        if year not in [2013, 2014, 2015]:
            holidays.update([
                datetime.date(year, 10, 5),   # Implantação da República (Republic Day)
                datetime.date(year, 11, 1),   # Dia de Todos os Santos (All Saints' Day)
                datetime.date(year, 12, 1),   # Restauração da Independência (Restoration of Independence)
                corpus_christi                # Corpo de Deus (Corpus Christi)
            ])


    return holidays

def add_temporal_features(df):
    """
    Adds time-based feature columns to the DataFrame based on the 'Date' column.
    """
    # Extract unique years from the dataset to calculate holidays dynamically
    unique_years = df['Date'].dt.year.unique()
    holidays_set = get_holidays(unique_years)

    new_features = {

        # 1=Monday to 7=Sunday
        'Weekday': df['Date'].dt.dayofweek + 1,

        # 1 to 24
        'Hour': df['Date'].dt.hour + 1,

        # 1 to 12
        'Month' : df['Date'].dt.month + 1,

        # True if Saturday (5) or Sunday (6)
        'Is_Weekend': df['Date'].dt.dayofweek >= 5,

        # True if the date is in the calculated holidays set
        # Evaluates national holidays dynamically. Essential for capturing anomalous consumption drops during public holidays
        'Is_Holiday': df['Date'].dt.date.isin(holidays_set)
    }
    
    
    df = pd.concat([df, pd.DataFrame(new_features)], axis=1)
    
    return df

## 3. Data Loading and Temporal Feature Engineering
We load the raw CSV. We apply the temporal features *before* melting the dataset. This is a crucial performance optimization: calculating the day of the week on 140,000 rows takes a fraction of a second, whereas doing it on 50 million rows (after melting) would take significantly longer.

In [3]:
# This cell should take less than 5 seconds to run
print("Loading data...")
df = pd.read_csv('../Datasets/Electricity Dataset.csv', sep=';', decimal=',')

# Rename timestamp column to 'Date' to keep naming consistent
df = df.rename(columns={'Unnamed: 0': 'Date'})
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by='Date', ascending=True).reset_index(drop=True)

# Add temporal features BEFORE melting (Huge performance boost: calculates on 140k rows instead of 51M)
print("Adding temporal features...")
df = add_temporal_features(df)

# Define the columns that should remain fixed (the time features)
# All other columns (the 370 clients) will be melted into rows
fixed_vars = ['Date', 'Weekday', 'Hour', 'Month', 'Is_Weekend', 'Is_Holiday']


# Show resulst
display(df)

Loading data...
Adding temporal features...


,Date,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,...,MT_366,MT_367,MT_368,MT_369,MT_370,Weekday,Hour,Month,Is_Weekend,Is_Holiday
0,2011-01-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,1,2,True,True
1,2011-01-01 00:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,1,2,True,True
2,2011-01-01 00:45:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,1,2,True,True
3,2011-01-01 01:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,2,2,True,True
4,2011-01-01 01:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,2,2,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140251,2014-12-31 23:00:00,2.538071,22.048364,1.737619,150.406504,85.365854,303.571429,11.305822,282.828283,68.181818,...,5.851375,697.102722,176.961603,651.026393,7621.621622,3,24,13,False,False
140252,2014-12-31 23:15:00,2.538071,21.337127,1.737619,166.666667,81.707317,324.404762,11.305822,252.525253,64.685315,...,9.947338,671.641791,168.614357,669.354839,6702.702703,3,24,13,False,False
140253,2014-12-31 23:30:00,2.538071,20.625889,1.737619,162.601626,82.926829,318.452381,10.175240,242.424242,61.188811,...,9.362200,670.763828,153.589316,670.087977,6864.864865,3,24,13,False,False
140254,2014-12-31 23:45:00,1.269036,21.337127,1.737619,166.666667,85.365854,285.714286,10.175240,225.589226,64.685315,...,4.095963,664.618086,146.911519,646.627566,6540.540541,3,24,13,False,False


## 4. Wide to Long Transformation (Melting)
The raw data has 370 columns (one for each client). To train a model effectively and engineer lag features properly, we "melt" the dataset so that each row represents a single observation for a single client at a specific time.

We also strictly downcast our numeric and categorical columns to save RAM.

In [ ]:
# ---------------------------------------------------------
# Wide to Long Transformation (Melt)
# ---------------------------------------------------------
df_long = pd.melt(
    df, 
    id_vars=fixed_vars,
    var_name='ClientID',
    value_name='Consumption'
)

# Free the wide-format dataframe (~500MB)
del df

# Memory optimization
df_long['Consumption'] = df_long['Consumption'].astype(np.float32)
df_long['ClientID']    = df_long['ClientID'].astype('category')

display(df_long)

## 5 National Weather
Since we don't know where each client is located, we built a national average temperature as a proxy signal.
Rather than a simple average of 4 cities, we weight each city by its population: Lisbon (547k), Porto (237k), Faro (64k), Évora (56k). 

From that weighted temperature we also derive two extra columns:

- HDH (Heating Degree Hours): how many degrees below 18°C it is. When this is high, people are heating their homes, so consumption goes up. 

- CDH (Cooling Degree Hours): how many degrees above 18°C it is. When this is high, people are running air conditioning, so consumption goes up. 

18°C is the standard threshold for Heating/Cooling Degree Days (HDD/CDD) in energy analytics

In [ ]:
# ---------------------------------------------------------
# Fetch hourly temperature from 4 Portuguese cities via Open-Meteo API
# Compute a population-weighted national average, then derive HDH/CDH
# ---------------------------------------------------------
cities = {
    "Lisbon": {"lat": 38.72, "lon": -9.14,  "population": 547},
    "Porto":  {"lat": 41.15, "lon": -8.61,  "population": 237},
    "Faro":   {"lat": 37.02, "lon": -7.93,  "population": 64},
    "Evora":  {"lat": 38.57, "lon": -7.91,  "population": 56},
}

total_population = sum(c["population"] for c in cities.values())
all_temps = []

for city, info in cities.items():
    print(f"  Fetching weather for {city}...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   info["lat"],
        "longitude":  info["lon"],
        "start_date": "2011-01-01",
        "end_date":   "2015-01-01",
        "hourly":     "temperature_2m",
        "timezone":   "Europe/Lisbon",
    }
    response = requests.get(url, params=params)
    data = response.json()

    weight = info["population"] / total_population
    temp_df = pd.DataFrame({
        "Date":          pd.to_datetime(data["hourly"]["time"]),
        "weighted_temp": np.array(data["hourly"]["temperature_2m"]) * weight,
    })
    all_temps.append(temp_df[["Date", "weighted_temp"]])

# ---------------------------------------------------------
# National average temperature (population-weighted)
# ---------------------------------------------------------
weather_df = all_temps[0][["Date"]].copy()
weather_df["Temp_National_Avg"] = sum(df["weighted_temp"] for df in all_temps).astype(np.float32)

# ---------------------------------------------------------
# Heating/Cooling Degree Hours (base 18 C, standard for energy modeling)
# ---------------------------------------------------------
weather_df["HDH"] = (18 - weather_df["Temp_National_Avg"]).clip(lower=0).astype(np.float32)
weather_df["CDH"] = (weather_df["Temp_National_Avg"] - 18).clip(lower=0).astype(np.float32)

# ---------------------------------------------------------
# Merge into df_long on the floored hour
# Temp_National_Avg is kept in the parquet for EDA but excluded from model
# regressors because Temp = 18 - HDH + CDH (perfect multicollinearity)
# ---------------------------------------------------------
weather_df["Date_Hour"] = weather_df["Date"]
df_long["Date_Hour"]    = df_long["Date"].dt.floor("h")

df_long = df_long.merge(
    weather_df[["Date_Hour", "Temp_National_Avg", "HDH", "CDH"]],
    on="Date_Hour",
    how="left"
).drop(columns=["Date_Hour"])

del all_temps, weather_df

print("Weather merge complete!")
display(df_long)

## 6. Handling Inactive Periods

To ensure our models learn from actual consumption behavior, we must remove periods of total inactivity:
1. **Leading Zeros (Pre-activation):** Many clients joined the grid after 2011. We identify the first active timestamp for each client and remove the preceding inactive period.
2. **Inactive Clients (Churn prevention):** We identify clients with near-zero consumption in the last month of the dataset and remove them entirely. Forecasting flat zeros for churned users would artificially distort error metrics like MAPE.

In [6]:
# This cell should take around 10 seconds to run

# PART 1 REMOVE TRIMMING LEADING ZEROS
print("Trimming leading zeros (finding actual start date for each client)...")

# Identify the first timestamp with consumption > 0 for each client
start_dates = df_long[df_long['Consumption'] > 0].groupby('ClientID', observed=True)['Date'].min().reset_index()
start_dates.columns = ['ClientID', 'StartDate']

# Merge the start dates back into the main dataframe
df_long = df_long.merge(start_dates, on='ClientID')

# Keep only the rows where Date is greater than or equal to the StartDate
df_long = df_long[df_long['Date'] >= df_long['StartDate']].copy()
df_long = df_long.drop(columns=['StartDate'])
print(f"Trimmed inactive periods. Remaining rows: {len(df_long)}")

# PART 2 REMOVE INACTIVE CLIENTS
print("Removing inactive clients (near 0 consumption in the last 90 days)...")

# Slelect the last month of data and calculate the total consumption per client
max_date = df_long['Date'].max()
cutoff_date = max_date - pd.Timedelta(days=30)
last_month_df = df_long[df_long['Date'] >= cutoff_date]
recent_consumption = last_month_df.groupby('ClientID')['Consumption'].sum()

# Identify active clients
active_clients = recent_consumption[recent_consumption > 1].index
inactive_count = len(recent_consumption) - len(active_clients)
print(f"Detected {inactive_count} inactive clients in the last month.")

# Keep only historically active clients
df_long = df_long[df_long['ClientID'].isin(active_clients)].copy()
df_long['ClientID'] = df_long['ClientID'].cat.remove_unused_categories()
print(f"Removed inactive clients. Remaining rows: {len(df_long)}")


# Show resulst
display(df_long)

Trimming leading zeros (finding actual start date for each client)...
Trimmed inactive periods. Remaining rows: 41936458
Removing inactive clients (near 0 consumption in the last 90 days)...
Detected 1 inactive clients in the last month.
Removed inactive clients. Remaining rows: 41796202


,Date,Weekday,Hour,Month,Is_Weekend,Is_Holiday,ClientID,Consumption,HDH,CDH
35040,2012-01-01 00:15:00,7,1,2,True,True,MT_001,3.807106,7.358518,0.0
35041,2012-01-01 00:30:00,7,1,2,True,True,MT_001,5.076142,7.358518,0.0
35042,2012-01-01 00:45:00,7,1,2,True,True,MT_001,3.807106,7.358518,0.0
35043,2012-01-01 01:00:00,7,2,2,True,True,MT_001,3.807106,7.518363,0.0
35044,2012-01-01 01:15:00,7,2,2,True,True,MT_001,5.076142,7.518363,0.0
...,...,...,...,...,...,...,...,...,...,...
51894715,2014-12-31 23:00:00,3,24,13,False,False,MT_370,7621.621582,11.229535,0.0
51894716,2014-12-31 23:15:00,3,24,13,False,False,MT_370,6702.702637,11.229535,0.0
51894717,2014-12-31 23:30:00,3,24,13,False,False,MT_370,6864.864746,11.229535,0.0
51894718,2014-12-31 23:45:00,3,24,13,False,False,MT_370,6540.540527,11.229535,0.0


## 7. Grouped Lag and Rolling Features

To forecast future consumption, the model needs to know recent past consumption. We engineer several time-shifted features to capture short, daily, and weekly seasonality profiles:
- **Lags:** What the client consumed 1 step ago (`Lag_15min`), 4 steps ago (`Lag_1h`), 1 day ago (`Lag_24h`), and 1 week ago (`Lag_1week`).
- **Rolling Windows:** The smoothed average consumption over the last 4 hours (`Rolling_Mean_4h`).

In [ ]:
# This cell should take less than 30 seconds to run

# ---------------------------------------------------------
# Sort chronologically per client before shifting
# ---------------------------------------------------------
print("Sorting data for lag calculations...")
df_long = df_long.sort_values(by=['ClientID', 'Date']).reset_index(drop=True)

# ---------------------------------------------------------
# Contiguity check: all timestamps should be exactly 15 minutes apart
# ---------------------------------------------------------
time_diffs = df_long.groupby('ClientID', observed=True)['Date'].diff()
irregular_gaps = time_diffs[(time_diffs.notnull()) & (time_diffs != pd.Timedelta(minutes=15))]
if len(irregular_gaps) != 0:
    print(f"Warning: Found {len(irregular_gaps)} missing or irregular timestamp intervals!")
    display(df_long.loc[irregular_gaps.index])

# ---------------------------------------------------------
# Consumption Lags (grouped by client to prevent cross-client bleed)
# ---------------------------------------------------------
print("Calculating consumption lags...")
df_long['Lag_15min'] = df_long.groupby('ClientID', observed=True)['Consumption'].shift(1).astype(np.float32)
df_long['Lag_1h']    = df_long.groupby('ClientID', observed=True)['Consumption'].shift(4).astype(np.float32)
df_long['Lag_24h']   = df_long.groupby('ClientID', observed=True)['Consumption'].shift(96).astype(np.float32)
df_long['Lag_1week'] = df_long.groupby('ClientID', observed=True)['Consumption'].shift(672).astype(np.float32)

print("Calculating rolling windows...")
df_long['Rolling_Mean_4h'] = df_long.groupby('ClientID', observed=True)['Consumption'].transform(
    lambda x: x.rolling(window=16).mean()
).astype(np.float32)

# ---------------------------------------------------------
# Weather Lags (thermal inertia: yesterday's temperature still affects today's demand)
# ---------------------------------------------------------
print("Calculating weather lags (24h thermal inertia)...")
df_long['HDH_lag24h'] = df_long.groupby('ClientID', observed=True)['HDH'].shift(96).astype(np.float32)
df_long['CDH_lag24h'] = df_long.groupby('ClientID', observed=True)['CDH'].shift(96).astype(np.float32)

# ---------------------------------------------------------
# Weather Anomaly (deviation from 30-day rolling seasonal baseline)
# A 10 C day in January is anomalously warm; the same 10 C in March is normal.
# High anomaly = unusual weather = bigger consumption spike.
# ---------------------------------------------------------
print("Calculating weather anomalies (30-day rolling baseline)...")
df_long['HDH_anomaly'] = (
    df_long['HDH'] - df_long.groupby('ClientID', observed=True)['HDH'].transform(
        lambda x: x.rolling(96 * 30, min_periods=96).mean()
    )
).astype(np.float32)

df_long['CDH_anomaly'] = (
    df_long['CDH'] - df_long.groupby('ClientID', observed=True)['CDH'].transform(
        lambda x: x.rolling(96 * 30, min_periods=96).mean()
    )
).astype(np.float32)

# ---------------------------------------------------------
# Drop NaN rows from lag/rolling operations
# ---------------------------------------------------------
print("Dropping NaNs...")
df_long = df_long.dropna().reset_index(drop=True)

print("Feature Engineering Complete!")
display(df_long)

## 8. Exporting the Processed Data
We save the result as a Parquet file. Parquet preserves our `float32` and `category` memory optimizations, drastically reducing both file size and load times for the modeling phase compared to a standard CSV.

In [8]:
display(df_long)

,Date,Weekday,Hour,Month,Is_Weekend,Is_Holiday,ClientID,Consumption,HDH,CDH,Lag_15min,Lag_1h,Lag_24h,Lag_1week,Rolling_Mean_4h
0,2012-01-08 00:15:00,7,1,2,True,False,MT_001,5.076142,8.141814,0.0,5.076142,19.035534,3.807106,3.807106,13.324873
1,2012-01-08 00:30:00,7,1,2,True,False,MT_001,5.076142,8.141814,0.0,5.076142,16.497461,3.807106,5.076142,12.531726
2,2012-01-08 00:45:00,7,1,2,True,False,MT_001,5.076142,8.141814,0.0,5.076142,5.076142,5.076142,3.807106,11.579949
3,2012-01-08 01:00:00,7,2,2,True,False,MT_001,5.076142,8.602323,0.0,5.076142,5.076142,3.807106,3.807106,10.786802
4,2012-01-08 01:15:00,7,2,2,True,False,MT_001,3.807106,8.602323,0.0,5.076142,5.076142,3.807106,5.076142,9.914340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41548229,2014-12-31 23:00:00,3,24,13,False,False,MT_370,7621.621582,11.229535,0.0,7189.188965,7891.892090,18378.378906,7081.081055,8192.567383
41548230,2014-12-31 23:15:00,3,24,13,False,False,MT_370,6702.702637,11.229535,0.0,7621.621582,7945.945801,17351.351562,6864.864746,8040.540527
41548231,2014-12-31 23:30:00,3,24,13,False,False,MT_370,6864.864746,11.229535,0.0,6702.702637,7351.351562,18864.865234,6918.918945,7929.054199
41548232,2014-12-31 23:45:00,3,24,13,False,False,MT_370,6540.540527,11.229535,0.0,6864.864746,7189.188965,17621.621094,6594.594727,7804.054199


In [9]:
# This cell should take less than 30 seconds to run

print("Saving to Parquet format...")
df_long.to_parquet('../Datasets/processed_electricity_data.parquet', index=False)
print("Preprocessing complete! File saved as '../Datasets/processed_electricity_data.parquet'")

Saving to Parquet format...
Preprocessing complete! File saved as '../Datasets/processed_electricity_data.parquet'
